In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score)

In [ ]:
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 22,
    'font.weight': 'normal',
    'axes.titleweight': 'bold'
})

In [ ]:
data_path = '/content/drive/My Drive/volleypred/data/'
plot_path = '/content/drive/My Drive/volleypred/plots/'

In [ ]:
df = pd.read_csv(data_path + 'df_model.csv')
df.head()

# Data splitting

In [ ]:
last_season = df['Season'].unique()[-1]
print(f"Last season (test set): {last_season}")

test_indices = df[df['Season'] == last_season].index
train_indices = df[df['Season'] != last_season].index

X = df.drop(columns=['Result', 'Season'])
y = df['Result']

X_train = X.loc[train_indices]
X_test = X.loc[test_indices]
y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

print(f"Number of matches in training set: {len(X_train)}")
print(f"Number of matches in test set: {len(X_test)}")

# Hyperparameter tuning

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

### Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=tscv, scoring='accuracy', n_jobs=-1)

print("Starting Random Forest tuning")
grid_rf.fit(X_train, y_train)

print(f"Best RF parameters: {grid_rf.best_params_}")
print(f"Best CV score: {grid_rf.best_score_:.4f}")

### XGBoost

In [ ]:
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss')

param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

grid_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb, cv=tscv, scoring='accuracy', n_jobs=-1)

print("Starting XGBoost tuning")
grid_xgb.fit(X_train, y_train)

print(f"Best XGB parameters: {grid_xgb.best_params_}")
print(f"Best CV score for XGB: {grid_xgb.best_score_:.4f}")

### Logistic Regression

In [ ]:
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2']
}

grid_lr = GridSearchCV(estimator=lr, param_grid=param_grid_lr, cv=tscv, scoring='accuracy', n_jobs=-1)

print("Starting Logistic Regression tuning")
grid_lr.fit(X_train, y_train)

print(f"Best LR parameters: {grid_lr.best_params_}")
print(f"Best CV score for LR: {grid_lr.best_score_:.4f}")

### SVM

In [ ]:
svc = SVC(probability=True, random_state=42)

param_grid_svc = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid_svc = GridSearchCV(estimator=svc, param_grid=param_grid_svc, cv=tscv, scoring='accuracy', n_jobs=-1)
print("Starting SVM tuning...")
grid_svc.fit(X_train, y_train)

print(f"Best SVM parameters: {grid_svc.best_params_}")
print(f"Best CV score for SVM: {grid_svc.best_score_:.4f}")

# Best models after training

In [ ]:
best_rf = grid_rf.best_estimator_
best_xgb = grid_xgb.best_estimator_
best_lr = grid_lr.best_estimator_
best_svc = grid_svc.best_estimator_

# Prediction on the test set

In [ ]:
y_pred_rf = best_rf.predict(X_test)
y_pred_xgb = best_xgb.predict(X_test)
y_pred_lr = best_lr.predict(X_test)
y_pred_svc = best_svc.predict(X_test)

# Confusion matrix

In [ ]:
def generate_confusion_matrix_final(y_true, y_pred, model_name,
                                   file_name=None):

    class_labels = ['3:0', '3:1', '3:2', '2:3', '1:3', '0:3']
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_labels,
        yticklabels=class_labels,
        cbar=True,
        annot_kws={"size": 12}
    )

    plt.xlabel('Predicted outcome')
    plt.ylabel('Actual outcome')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig(plot_path + file_name, dpi=300)
    plt.show()

    print(f"Plot saved to: {plot_path}")

In [ ]:
models_preds = {
    'Random Forest': y_pred_rf,
    'XGBoost': y_pred_xgb,
    'Logistic Regression': y_pred_lr,
    'SVM': y_pred_svc
}

for name, y_pred in models_preds.items():
    print(f"Generating confusion matrix for {name}...")
    generate_confusion_matrix_final(y_test, y_pred, model_name=name,
                                    file_name=f'confusion_matrix_{name}.png')
#

# Metrics

In [ ]:
metrics_list = []

for name, y_pred in models_preds.items():
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average='macro')
    f1_macro = f1_score(y_test, y_pred, average='macro')

    metrics_list.append({
        'Model': name,
        'Accuracy': acc,
        'Balanced Acc': bal_acc,
        'Precision (Macro)': prec_macro,
        'F1-Score (Macro)': f1_macro
    })

df_metrics = pd.DataFrame(metrics_list)
df_metrics = df_metrics.round(4)
df_metrics = df_metrics.sort_values(by='Accuracy', ascending=False)

display(df_metrics)